In [1]:
# import os # этой библиотекой переодически смотрю файлики в области видимости jupyter
import pandas as pd # для предварительного анализ
import json # первое что пришло на ум для сериализации json
from functools import reduce # еще один вариант решения который попробую, но скорее всего не самое лучшее
import sys # пригодиться для анализа

In [2]:
!python --version

Python 3.9.0


# ДЗ 1

## Предварительный анализ

In [3]:
# os.listdir()

Из личного интереса решил немного попробовать поиграться инструментом профилировки в рамках ДЗ

In [4]:
pip install memory_profiler

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\users\qwerty\desktop\hw\venv\scripts\python.exe -m pip install --upgrade pip' command.


In [5]:
%load_ext memory_profiler

Предварительно изучил данные, чтобы убедиться, что словарь из правильного количества ключей.

In [6]:
df = pd.read_json('purchase_log.txt', lines=True)

In [7]:
df = df[1:] # пропускаем строку заголовка

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99999 entries, 1 to 99999
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   user_id   99999 non-null  object
 1   category  99999 non-null  object
dtypes: object(2)
memory usage: 1.5+ MB


In [9]:
df.head()

,user_id,category
1,1840e0b9d4,Продукты
2,4e4f90fcfb,Электроника
3,afea8d72fc,Электроника
4,373a6055fe,Бытовая техника
5,9b2ab046f3,Электроника


In [10]:
len(df['user_id'].unique()) # необходимо для валидации результата

99517

Так как задача по сути сведения к хэш таблице неких логов, то число - 99517 потребуется для сравнения результатов.

In [11]:
len(df['category'].unique()) # факультативная статистика, личный интерес

7

In [12]:
# еще одна факультативная статистика, личный интерес

In [13]:
category_df = df.groupby('category')['user_id'].count().reset_index().sort_values(by='user_id', ascending=False)
category_df

,category,user_id
5,Электроника,33466
0,Бытовая техника,23670
2,Продукты,14010
1,Досуг,11783
3,Строительство и ремонт,9009
4,Товары для животных,7028
6,не определена,1033


In [14]:
category_df['category'].unique()

array(['Электроника', 'Бытовая техника', 'Продукты', 'Досуг',
       'Строительство и ремонт', 'Товары для животных', 'не определена'],
      dtype=object)

Так как количество строк 99999 и количество пользователй 99517, то имеет смысл сделать агрегат по user_id и посмотреть, какие категории повторяются

In [15]:
data_agg = df.groupby('user_id')['category'].agg(','.join)

In [16]:
data_agg = df.groupby('user_id')['category'].agg(
    categories=', '.join,
    count='count',
)

In [17]:
df_gte2 = (data_agg[data_agg['count'] >= 2]).reset_index()
df_gte2

,user_id,categories,count
0,0061aaf3f9,"Продукты, Электроника",2
1,02a3642b58,"Электроника, Строительство и ремонт",2
2,02ce3f6ba4,"Досуг, Бытовая техника",2
3,02fd760e37,"Бытовая техника, Строительство и ремонт",2
4,03787309eb,"Строительство и ремонт, Электроника",2
...,...,...,...
476,fd3c2d449c,"Бытовая техника, Строительство и ремонт",2
477,fe94c62069,"Электроника, Бытовая техника",2
478,ffb4426f85,"Досуг, Электроника",2
479,ffb83955b3,"Электроника, Продукты",2


In [18]:
data_agg['count'].sum() # никаких аномалий при группировке нет, ничего не дропнулось, все данные на месте

np.int64(99999)

В случае хэш такблиц(словарь), пользователи имеющие более одной категории будут перетирать значения по мере выполнения цикла.
ДЗ звучит так буд-то это и подразумевалос.
Дополнительно приложу еще свой вариант решения.

## Решение

In [19]:
def parse_log1():
    purchase_dict = {}
    with open('purchase_log.txt',  mode='r',  encoding='utf-8',) as f:
        f.readline()
        for l in f:
            data = json.loads(l)
            purchase_dict[data['user_id']] = data['category']
    return purchase_dict

In [20]:
def parse_log2():
    purchase_dict = {}
    with open('purchase_log.txt',  mode='r',  encoding='utf-8',) as f:
        f.readline()
        purchase_dict = dict(
            map(
                lambda d: (d['user_id'], d['category']),
                map(json.loads, f)
               )
        )
    return purchase_dict
    

In [21]:
def parse_log3():
    purchase_dict = {}
    with open('purchase_log.txt',  mode='r',  encoding='utf-8',) as f:
        f.readline()
        data = f.read().splitlines()
        # пилим строку. Должно работать быстрее, но когнитивно тяжелее воспринимать
        purchase_dict = {
                str.partition(str.partition(l, '"user_id": "')[2], '"')[0]:
                str.partition(str.partition(l, '"category": "')[2], '"')[0]
                for l in data
        }
    return purchase_dict

In [22]:
def parse_log4():
    purchase_dict = {}
    with open('purchase_log.txt',  mode='r',  encoding='utf-8',) as f:
        f.readline()  
        return reduce(
            lambda acc, d: acc.update({d['user_id']: d['category']}) or acc,
            map(json.loads, f),
            {}, 
        )
    

##  Профилирование (личное любопытство)

In [23]:
%memit pass # общее потребление Python

peak memory: 159.85 MiB, increment: 0.06 MiB


In [24]:
%timeit -n 2 -r 5 parse_log1()

148 ms ± 1.76 ms per loop (mean ± std. dev. of 5 runs, 2 loops each)


In [25]:
%memit -r 10 parse_log1() 

peak memory: 163.07 MiB, increment: 2.53 MiB


In [26]:
%timeit -n 2 -r 5 parse_log2()

163 ms ± 8.29 ms per loop (mean ± std. dev. of 5 runs, 2 loops each)


In [27]:
%memit -r 10 parse_log2()  

peak memory: 163.09 MiB, increment: 2.52 MiB


In [28]:
%timeit -n 2 -r 5 parse_log3()

74.3 ms ± 1.2 ms per loop (mean ± std. dev. of 5 runs, 2 loops each)


In [29]:
%memit -r 10 parse_log3()  

peak memory: 177.18 MiB, increment: 15.14 MiB


In [30]:
%timeit -n 2 -r 5 parse_log4()

155 ms ± 3.5 ms per loop (mean ± std. dev. of 5 runs, 2 loops each)


In [31]:
%memit -r 10 parse_log4() 

peak memory: 164.60 MiB, increment: 2.54 MiB


## Анализ результатов

In [32]:
p1 = parse_log1()
p2 = parse_log2()
p3 = parse_log3()
p4 = parse_log4()

In [33]:
p1 == p2 == p3 == p4

True

In [34]:
len(p1.keys())

99517

результат сопоставим с количеством пользователей из user_id

 - parse_log1 - лучший по когнитивной нагрузке
 - parse_log3 - самый быстрый, но вероятно легко сломается при дополнении структуры json, новыми параметрами. Особенно если делитель строки будет попадать шаблоном в них
 - parse_log2 и parse_log4 - в данном случае бестолковые способы, так как тяжелее воспринимать, дебажить, дополнять.

## Вывод результатов(для фиксации выполнения)

In [35]:
p1

{'1840e0b9d4': 'Продукты',
 '4e4f90fcfb': 'Электроника',
 'afea8d72fc': 'Электроника',
 '373a6055fe': 'Бытовая техника',
 '9b2ab046f3': 'Электроника',
 '9f39d307c3': 'Электроника',
 '44edeffc91': 'Продукты',
 '704474fa2d': 'Продукты',
 '1de31be403': 'Бытовая техника',
 'b71f36a5e4': 'Продукты',
 '79843a685a': 'Продукты',
 'ff68cee0d6': 'Бытовая техника',
 'e8447c40e2': 'Досуг',
 '98d290be27': 'Электроника',
 'fa0079a5a8': 'Досуг',
 '22d2f03a17': 'Досуг',
 '3f8e1ccd3f': 'Электроника',
 '81a9988b83': 'Электроника',
 '65f44a2eb7': 'Досуг',
 '5f8fbb0149': 'Бытовая техника',
 '3ecff691fd': 'Электроника',
 'dd0e912251': 'Электроника',
 '9250593d55': 'Электроника',
 '37c4e090e4': 'Электроника',
 '22fa3ea76f': 'Электроника',
 '73b876b237': 'Электроника',
 'f983a69d67': 'Товары для животных',
 '4ea17212f8': 'Продукты',
 'c6de96a5e2': 'Товары для животных',
 'd8e3bcee53': 'Продукты',
 '680aa815f9': 'Продукты',
 '4a92a68cf9': 'Электроника',
 '905e60cb52': 'Бытовая техника',
 '393bcfb298': 'Продук

In [36]:
for i, (user_id, category) in enumerate(p1.items()):
    if i >= 2:
        break
    print(f'{user_id} ‘{category}‘')

1840e0b9d4 ‘Продукты‘
4e4f90fcfb ‘Электроника‘


## Дополнительный разбор в случае дублей пользователей

В блоке анализа мной найдены дубликаты идентификаторов пользователей. 
Для логов вполне ожидаемо иметь дубли, и скорее всего имеет смысл обработать их иначе. 
Ниже представлен контрольный пользователь `0061aaf3f9`, который имеет категории: Продукты, Электроника.
Но в связи с особенностями слоаварей, перезатирается. В условии задачи ничего не сказано по этому поводую

In [37]:
df_gte2.iloc[0:] # категории на самом деле

,user_id,categories,count
0,0061aaf3f9,"Продукты, Электроника",2
1,02a3642b58,"Электроника, Строительство и ремонт",2
2,02ce3f6ba4,"Досуг, Бытовая техника",2
3,02fd760e37,"Бытовая техника, Строительство и ремонт",2
4,03787309eb,"Строительство и ремонт, Электроника",2
...,...,...,...
476,fd3c2d449c,"Бытовая техника, Строительство и ремонт",2
477,fe94c62069,"Электроника, Бытовая техника",2
478,ffb4426f85,"Досуг, Электроника",2
479,ffb83955b3,"Электроника, Продукты",2


In [38]:
p1['0061aaf3f9'] # из функции парсера

'Электроника'

Первое что пришло на ум для наглядности использовать структуру ввида:

In [39]:
{
    '000117a2a3': ['Электроника', '', '', '', '', '', ''],
    '0002983af3': ['Электроника', '', '', '', '', '', '']
}

{'000117a2a3': ['Электроника', '', '', '', '', '', ''],
 '0002983af3': ['Электроника', '', '', '', '', '', '']}

Данная структура, на мой взгляд, не самая оптимальная в для хранения и обработки данных, но моя цель обозначить, 
особенности обработки логов с использование словарей. Разумеется можно элементы массива кодировать более маленькими по размеру типами данных, 
например int или bool и в зависимости от позиции мапить с категорией - что вполне корректно применять в СУБД и не только.
Или же вообще использовать иную структуру, или даже несколько структур в разных переменных, в рамках применения задач инженера.
Но таковых решений может быть так же много как и самих задач.

Где каждый индекс массива имеет смысл категории:

In [40]:
categories = ['Электроника', 'Бытовая техника', 'Продукты', 'Досуг',
              'Строительство и ремонт', 'Товары для животных', 'не определена']


cat_pos = {c: i for i, c in enumerate(categories)}
cat_pos

{'Электроника': 0,
 'Бытовая техника': 1,
 'Продукты': 2,
 'Досуг': 3,
 'Строительство и ремонт': 4,
 'Товары для животных': 5,
 'не определена': 6}

In [41]:
# import numpy as np

# 1. Позиция каждой строки
df['pos'] = df['category'].map(cat_pos)

# 2. Оставить валидные
valid = df.dropna(subset=['pos']).copy()
valid['pos'] = valid['pos'].astype(int)

# 3. Оставить по одной строке на (user_id, pos)
pairs = valid[['user_id', 'pos']].drop_duplicates()

def _row_from_positions(positions, categories):
    row = [''] * len(categories)
    for p in positions:
        row[int(p)] = categories[int(p)]
    return row

# 4. Собрать словарь через groupby + apply
result = (
    pairs.groupby('user_id')['pos']
         .apply(lambda s: _row_from_positions(s, categories))
         .to_dict()
)



In [42]:
result

{'000117a2a3': ['Электроника', '', '', '', '', '', ''],
 '0002983af3': ['Электроника', '', '', '', '', '', ''],
 '0002b641bc': ['', 'Бытовая техника', '', '', '', '', ''],
 '000302625b': ['Электроника', '', '', '', '', '', ''],
 '0003962bbf': ['', '', '', 'Досуг', '', '', ''],
 '0003c135ea': ['', 'Бытовая техника', '', '', '', '', ''],
 '0003dea8e2': ['', '', '', '', 'Строительство и ремонт', '', ''],
 '00043e4c99': ['', '', '', 'Досуг', '', '', ''],
 '0004cbc893': ['', '', 'Продукты', '', '', '', ''],
 '0004ed9c1c': ['Электроника', '', '', '', '', '', ''],
 '0005a075bc': ['', '', 'Продукты', '', '', '', ''],
 '0005a1df4b': ['Электроника', '', '', '', '', '', ''],
 '0007b17b9c': ['', 'Бытовая техника', '', '', '', '', ''],
 '0007b4f64a': ['', 'Бытовая техника', '', '', '', '', ''],
 '0009a1123d': ['', 'Бытовая техника', '', '', '', '', ''],
 '0009df2bdb': ['', '', '', '', '', 'Товары для животных', ''],
 '000a12712b': ['', '', '', '', 'Строительство и ремонт', '', ''],
 '000a8b81fe': [

Да следует отметить изменился порядок. Но поскольку это хэш таблица с доступом по ключу O(1), то порядок не важен.

In [43]:
result.get('0061aaf3f9')

['Электроника', '', 'Продукты', '', '', '', '']

# ДЗ 2

В условии задачи сказано что чанк не должен помещаться полностью в ОЗУ, и нужно считывать построчно. 
Применю похожее условие и для предварительного анализа.

## Предварительный просмотр исходных данных

In [44]:
with open('visit_log.csv', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 5 : break
        line = line.strip()
        # обработка строки
        print(line)

user_id,source
6450655ae8,other
b4ea53e670,other
0,other
96064ae9e0,other


Примерно понятна структура данных. Далее я постарлся избежать подключения сторонних библиотек для выполнения задачи. В реальной боевой системе допускаю появления спец символов, которых в ДЗ нет.

## Решение

В этот раз не оформлял код в виде функции

In [45]:
with \
    open('visit_log.csv', 'r', encoding='utf-8') as f_visit, \
    open('funnel.csv', 'w', encoding='utf-8', newline='') as f_funnel:
    f_visit.readline() # сдвигаю курсор на заголовок
    f_funnel.write('user_id,source,category\n')
    for i, line_visit in enumerate(f_visit):
        user_id, source = line_visit.strip().split(',')
        if user_id in p1:
            f_funnel.write(f'{user_id},{source},{p1[user_id]}\n')

In [46]:
with open('funnel.csv', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3 : break
        line = line.strip()
        # обработка строки
        print(line)

user_id,source,category
1840e0b9d4,other,Продукты
4e4f90fcfb,context,Электроника


## Решение с учетом дублей из аналитического блока ДЗ 1

Если использовать решение, которое учитывает несколько категорий, то значительно решение не измениться.

В качестве контрольного пользователя беру идентификатор: `0061aaf3f9`

In [47]:
with \
    open('visit_log.csv', 'r', encoding='utf-8') as f_visit, \
    open('funnel_2.csv', 'w', encoding='utf-8', newline='') as f_funnel:
    f_visit.readline() # сдвигаю курсор на заголовок
    f_funnel.write('user_id,source,categories\n')
    for i, line_visit in enumerate(f_visit):
        user_id, source = line_visit.strip().split(',')
        if user_id in p1:
            f_funnel.write(f'{user_id},{source},{";".join([i for i in result[user_id] if i != ""])}\n')

In [48]:
with open('funnel_2.csv', 'r', encoding='utf-8') as f_funnel_2:
    head = f_funnel_2.readline().strip()
    print(head)
    for i, line_funnel in enumerate(f_funnel_2):
        user_id, source, categories = line_funnel.strip().split(',')
        if '0061aaf3f9' == user_id:
            print(f'{user_id}, {source}, {categories}')

user_id,source,categories
0061aaf3f9, other, Электроника;Продукты
0061aaf3f9, context, Электроника;Продукты
